### **[The Model That Freezes Your Drawdown](https://medium.com/@Kryptera/41f1289e03c4)**

> *Richard Fabian’s 39-week switch, tested from 1992 to 2026*

In [1]:
!pip install -qq vectorbt "pandas==2.2.3" "numba==0.65.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.7/451.7 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.5/316.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 104.6 MB/s eta 0:00:00


In [2]:
import sys

import warnings
warnings.filterwarnings('ignore')

from IPython.display import clear_output, display

import numpy as np
import pandas as pd

import yfinance as yf
import vectorbt as vbt

IN_COLAB = 'google.colab' in sys.modules

np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1200)
pd.set_option('display.precision', 3)
pd.set_option('display.float_format', '{:.3f}'.format)

clear_output()
%autosave 15

Autosaving every 15 seconds


In [3]:
TICKERS = ["^GSPC", "^DJI", "^DJU"]
TRADED = "^GSPC"
START, END = "1970-01-01", "2030-01-01"

MA_WEEKS = 39
MIN_BELOW_TO_SELL = 2
INIT_CASH = 100_000
FEES, SLIPPAGE = 0.0001, 0.0002

In [4]:
# -------------------------
# Data
# -------------------------
raw = yf.download(TICKERS, start=START, end=END, interval="1d", group_by="ticker", auto_adjust=True, prepost=False)

close_d = pd.DataFrame({t: raw[t]["Close"] for t in TICKERS}).dropna()
open_d = pd.DataFrame({t: raw[t]["Open"] for t in TICKERS}).loc[close_d.index]

# Yahoo's index Opens are frequently just a copy of the prior Close (^GSPC: ~96% of
# 2000-2004 bars). The damage is much smaller here than in a daily mean-reversion
# system — this model holds for months, and the weekly Open is a Monday open, so a
# stale value costs at most one day of drift per trade. Still, know the number.
stale = (np.abs(open_d[TRADED] - close_d[TRADED].shift(1)) < 1e-8).mean() * 100
print(f"[data audit] {TRADED}: Open == prior Close on {stale:.1f}% of daily bars")
print("             swap TRADED to 'SPY' if you want fills at a price that traded.\n")

[*********************100%***********************]  3 of 3 completed

[data audit] ^GSPC: Open == prior Close on 38.1% of daily bars
             swap TRADED to 'SPY' if you want fills at a price that traded.



In [5]:
# -------------------------
# Weekly resample (W-FRI = week ending Friday)
# -------------------------
close_w = close_d.resample("W-FRI").last().dropna()
open_w = open_d.resample("W-FRI").first().loc[close_w.index]

sma_w = close_w.rolling(MA_WEEKS).mean()
above = close_w > sma_w
below = close_w < sma_w

all_above = above.all(axis=1)
two_or_more_below = below.sum(axis=1) >= MIN_BELOW_TO_SELL

In [6]:
# -------------------------
# Signals — vectorbt keeps the state between a buy and the next sell
# -------------------------
entries = all_above.shift(1).fillna(False).astype(bool)
exits = two_or_more_below.shift(1).fillna(False).astype(bool)

pf = vbt.Portfolio.from_signals(
  close=open_w[TRADED],
  entries=entries.to_numpy(),
  exits=exits.to_numpy(),
  init_cash=INIT_CASH,
  fees=FEES,
  slippage=SLIPPAGE,
  freq="7D",
)

pf_bh = vbt.Portfolio.from_holding(close_w[TRADED], init_cash=INIT_CASH, freq="7D")

In [7]:
# -------------------------
# Time in market
# -------------------------
e, x = entries.to_numpy(), exits.to_numpy()
mask = np.zeros(len(e), dtype=bool)
state = False
for i in range(len(e)):
  if e[i]:
    state = True
  if x[i]:
    state = False
  mask[i] = state

print("\n=== Fabian Timing Model (weekly) ===")
print(f"Universe      : {', '.join(TICKERS)}")
print(f"Time in market: {mask.mean() * 100:.2f}%  ({mask.sum()} of {len(mask)} weeks)\n")
display(pd.concat([pf.stats().rename("Fabian"), pf_bh.stats().rename("Buy & Hold")], axis=1))

fig = pf.plot(
  subplots=["value", "underwater", "orders", "trade_pnl"],
  title=f"Fabian Timing Model",
  make_subplots_kwargs=dict(row_heights=[0.30, 0.20, 0.20, 0.30], vertical_spacing=0.03),
)
fig.update_layout(height=1200)
fig.show()


=== Fabian Timing Model (weekly) ===
Universe      : ^GSPC, ^DJI, ^DJU
Time in market: 66.30%  (1200 of 1810 weeks)



,Fabian,Buy & Hold
Start,1992-01-03 00:00:00,1992-01-03 00:00:00
End,2026-09-04 00:00:00,2026-09-04 00:00:00
Period,12670 days 00:00:00,12670 days 00:00:00
Start Value,100000.000,100000.000
End Value,746907.965,1828239.640
Total Return [%],646.908,1728.240
Benchmark Return [%],1745.795,1728.240
Max Gross Exposure [%],100.000,100.000
Total Fees Paid,2701.608,0.000
Max Drawdown [%],25.709,56.244
